Lab 19: API Requests and Caching — Analysis Run these experiments to see the difference caching makes.

Before you start: Make sure your crypto.py implementations pass all tests.

Setup: Put your CoinGecko Demo API key in the cell below.

In [1]:
import sys
sys.path.insert(0, '../src')

import time
from crypto import get_price, get_prices_batch, CoinCache, get_price_cached

API_KEY = "CG-Ba8eKQ5uVTbFFuLs6QS9pcfk"  # Replace with your CoinGecko Demo key


Experiment 1: Uncached vs. Cached Fetch Bitcoin's price 10 times — first without caching, then with caching. Compare the total time and number of API calls.

In [2]:
# --- Uncached: 10 direct API calls ---
start = time.time()
for i in range(10):
    price = get_price("bitcoin", API_KEY)
uncached_time = time.time() - start

print(f"Uncached: 10 requests in {uncached_time:.2f} seconds")
print(f"Last price: ${price:,.2f}")

Uncached: 10 requests in 0.71 seconds
Last price: $72,048.00


In [3]:
# --- Cached: 10 lookups through cache ---
cache = CoinCache(ttl_seconds=60)

start = time.time()
for i in range(10):
    price = get_price_cached("bitcoin", API_KEY, cache)
cached_time = time.time() - start

print(f"Cached: 10 lookups in {cached_time:.2f} seconds")
print(f"Cache hits: {cache.hits}, Cache misses: {cache.misses}")
print(f"Speedup: {uncached_time / cached_time:.1f}x faster")

Cached: 10 lookups in 0.07 seconds
Cache hits: 9, Cache misses: 1
Speedup: 9.6x faster


Writeup Questions — Experiment 1 How many API calls did the cached version actually make? Why that number? What was the approximate speedup? Why is the difference so large? Is there any downside to this speedup? What are you giving up? Your answers here:
1. It made one api call. This happened becasue on this first lookup bitcoin wasnt in the cache yet so it ends up calling the API and stores those results 
2. The speedup was 9.6x faster. This difference is so big because of all the network latency. Each of the APi calls takes up alot of latency. 
3. Yes there is. The tradeoff is the data freshness, or staleness. What I mean is you never know if the cache price is still accurate which can be a big problem. So I would say thats the tradeoff. 

Experiment 2: TTL Exploration Try three different TTL values. For each one, simulate a pattern of lookups spaced 2 seconds apart and observe the hit rate.

In [4]:
ttl_values = [1, 5, 30]

for ttl in ttl_values:
    cache = CoinCache(ttl_seconds=ttl)

    for i in range(6):
        price = get_price_cached("bitcoin", API_KEY, cache)
        if i < 5:  # Don't sleep after last lookup
            time.sleep(2)

    total = cache.hits + cache.misses
    hit_rate = cache.hits / total * 100
    print(f"TTL={ttl:2d}s: {cache.hits} hits, {cache.misses} misses, hit rate={hit_rate:.0f}%")

TTL= 1s: 0 hits, 6 misses, hit rate=0%
TTL= 5s: 4 hits, 2 misses, hit rate=67%
TTL=30s: 5 hits, 1 misses, hit rate=83%


Writeup Questions — Experiment 2 With TTL=1 second and lookups every 2 seconds, what hit rate do you expect? Did the results match? If you were building a portfolio tracker that updates every time you open the app, what TTL would you choose? Explain your reasoning. Is there a scenario where you'd want a TTL of 0 (no caching at all)? What about a TTL of infinity (cache forever)? Your answers here:
1. When TTL is 1s and the lookups are every 2s that leaves the cache to expire before the next lookup even happens. This leads you to expect 0% hits, which is what the data says. 
2. TTL=5-30s is good for a portfolie tracker id say. It gives the high hit rates, yet it still keeps the prices pretty decently fresh.
3. I would say TTL=0 is nice for real-time traders. Like day traders who need the see the latest pricees at all times so they know when to cloes, or sell, or buy when they want, and its accurate enough. TTL=inifity I would say isnt the most effective, but its good for historical data, or data that doesnt really change. 

Experiment 3: Batch Efficiency Compare fetching 5 coins one at a time vs. in a single batch request.

In [5]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "dogecoin"]

# --- Individual calls ---
start = time.time()
individual_prices = {}
for coin in coins:
    individual_prices[coin] = get_price(coin, API_KEY)
individual_time = time.time() - start

print(f"Individual: {len(coins)} calls in {individual_time:.2f} seconds")

# --- Batch call ---
start = time.time()
batch_prices = get_prices_batch(coins, API_KEY)
batch_time = time.time() - start

print(f"Batch: 1 call in {batch_time:.2f} seconds")
print(f"Speedup: {individual_time / batch_time:.1f}x faster")

# Show the prices
print("\nPrices:")
for coin, price in batch_prices.items():
    print(f"  {coin}: ${price:,.2f}")

Individual: 5 calls in 0.63 seconds
Batch: 1 call in 0.10 seconds
Speedup: 6.1x faster

Prices:
  bitcoin: $71,796.00
  ethereum: $2,206.33
  solana: $83.54
  cardano: $0.25
  dogecoin: $0.09


Writeup Questions — Experiment 3

How much faster was the batch call? Where does that time saving come from?
Batching and caching are both ways to reduce API calls. When would you use one vs. the other? Can you use both?
Your answers here: 
1. The batch call was 6.1x faster becaseu one api call with 5 coins ends up elimitaing the network, that you started by paying 5 seperate times. It makes you do one round trip, instead of five different trips. 
2. I would say use batch when you are trying to get more than one coins at one time, and you dont mind a little bit of out of date, or older data. Yes you can use both. You can use both for your advantage to. Use bach to fetch multiple coins, then serve repeated request from cache. 